In [1]:
import pandas as pd
import csv
import pandas_gbq
import time
from datetime import datetime
import os
from google.oauth2 import service_account
from google.cloud import bigquery
import numpy as np
import re

In [2]:
CREDS = '../../converge-database-0331482f2ee5.json'
client = bigquery.Client.from_service_account_json(json_credentials_path=CREDS)

In [3]:
valdate =pd.to_datetime('2026-03-31')

In [4]:
month ='202603'

In [5]:
query =f'''

DECLARE termPlans ARRAY<STRING> DEFAULT ['19', '41', '42', '43', '44', '47', '92', '93', '94', '96', '40'];
DECLARE valCodePlans ARRAY<STRING> DEFAULT ['00235', '00519', '00595', '00597', '00695', '00725', '00794', '00795', 'P0302', 'P0395', '0075N'];

WITH field_prep AS (

  select policyno, pmdplan as plan,pmdissuedt, 
        wrvage, wrvduratn, 
        CASE 
          WHEN CAST(wrvduratn AS INTEGER) = 0 OR pmdstssub = 'ETI' 
              THEN GREATEST(0, wrvage - CEIL(DATE_DIFF(DATE('{valdate}'), DATE(pmdissuedt), DAY) / 365.0))
          ELSE wrvage
      END AS issAge,


        CASE 
          WHEN LEFT(pmdplan,1) = 'F' THEN 'F'  
          WHEN s.catmkt = 'H' THEN 'H'         
          WHEN s.catmkt = 'P' THEN 'P'         
          ELSE '0'  
      END AS catmkt_formula,

        CEIL(
              DATE_DIFF(
                      DATE("{valdate}")
                      , 
                      DATE(pmdissuedt), DAY) / 365.0) as duratn, 
        pmdstssub, 
        wrvpbfsex,
        s.catmkt, v.qx_new, 
        v.mort_char, v.intchar, 
        s.pmdmode,s.pmdplan,
        d.alfa_credit_method, 
        (
          CASE 
              WHEN d.product2 IS NULL THEN "0" 
              ELSE d.product2 END
        ) as product2,
        pmdstsexdt, final_famt,
        prmannl, amtreserve, 
        pmdplanver,pbfrplanvr,
        d.bonus, d.alfa_dbinc, 
        pbfrplan, d.past_growth_identifier, 
        dg.year,dg.rate,
        pmddtdwd,t.length as lengths,
        s.exhibit5,
        v.pp,
        LPAD(CAST(s.valcode AS STRING), 5, '0') as valcode,
        t.term_plans,

        CASE 
          WHEN (product2 = "0" OR product2 IS NULL) THEN 1
          WHEN product2 ='Simple' THEN 1+alfa_dbinc*(DATE_DIFF(DATE("{valdate}"),pmdissuedt, YEAR))
          WHEN product2 ='Missing' THEN POWER((1+alfa_dbinc),CAST((CEIL(DATE_DIFF(DATE('{valdate}'), DATE(pmdissuedt), DAY) / 365.0)-1) as INT64))
        ELSE rate  END past_growth,

        CASE 
          WHEN pmddtdwd is NULL THEN DATE("{valdate}")
          WHEN pmddtdwd is not NULL THEN  pmddtdwd
        END as paidDate,

  from `life_prod.seriatim` s
  LEFT JOIN `life_prod.inforce_lookups` v ON v.valcode_new =LPAD(CAST(s.valcode AS STRING), 5, '0')

  LEFT JOIN `life_prod.dividend` d ON d.plan_ver = CONCAT(LEFT(s.pmdplan,4), RIGHT(s.pmdplanver,1)) AND d.added_month <="{month}"
  LEFT JOIN (SELECT * from `life_prod.div_growth` WHERE inforce_year = {int(month[:4])} ) dg ON dg.year = CONCAT(EXTRACT(YEAR FROM pmdissuedt), "_", LOWER(d.product2))
  LEFT JOIN `life_prod.termplans` t ON t.term_plans = s.pmdplan
  WHERE set_month ="{month}" AND policyno IN (SELECT pmdno from `lifetemp.seriatim` WHERE set_month ='202512') ),
  policy_count as (
    select policyno, count(*) as pol_ct from `life_prod.seriatim`
    GROUP BY policyno
    HAVING pol_ct >1
  )
#Make sure duplicate valcode is fixed for these valcodes. "0099H", "199-5", "299-5"
SELECT 
  plan,
  term_plans,
  issAge,
  pp as prem_check_field,
  wrvpbfsex as gender,

  "N" as class,

  catmkt_formula as catmkt,

  mort_char,
  valcode,
  intchar,

  RIGHT(pmdmode,1) as mode,

  CASE 
      WHEN alfa_credit_method IS NULL THEN "N"
      ELSE alfa_credit_method
  END as bnscreditMtd,

  CASE 
      WHEN product2 ='CPI'THEN 'G'
      ELSE "N"
  END as guar_non_guar,

  #Need different approach
  CASE 
        WHEN COUNT(*) OVER (PARTITION BY policyno) = 1 THEN 1
        WHEN ROW_NUMBER() OVER (
                PARTITION BY policyno 
                ORDER BY amtreserve DESC
             ) = 1 THEN 1
        ELSE 0
    END AS uniq_policy_check,


  CASE 
      WHEN pmdstssub='ETI' THEN 'T'
      WHEN pbfrplanvr is NOT NULL THEN 'T'
      WHEN plan IN ('19', '41', '42', '43', '44', '47', '92', '93', '94', '95', '40') THEN 'T'
      ELSE 'W'
  END as term_wl,
  

  EXTRACT(YEAR FROM pmdissuedt) as issyr,

  CASE 
      WHEN pmdstssub='ETI' THEN EXTRACT(MONTH FROM pmdstsexdt)
      ELSE EXTRACT(MONTH FROM pmdissuedt) 
  END AS issMonth,

  "E" as newBusinessInd,
  CONCAT(LEFT(pmdplan,4), RIGHT(pbfrplanvr,1)) as planver,
  pmdplanver,
  product2,
  CONCAT(EXTRACT(YEAR FROM pmdissuedt), "_", LOWER(product2)) as test,
  past_growth,
  duratn,
  alfa_dbinc,
  pmdissuedt,
  #Done


  CASE WHEN (
              (final_famt/past_growth)/1000 <0.00011 ) THEN 0.00011
      ELSE (final_famt/past_growth)/1000
  END as FixedLives,
  
  1 as PolCount,
  final_famt as face,
  
  (
    CASE 
        WHEN pmdstssub IN ('PRMPY', "WOD") THEN 1
        ELSE 0
    END
  )*prmannl as PremIf,
  
    CASE WHEN (
                (final_famt/past_growth)/1000 <0.00011 ) THEN 0.00011
        ELSE (final_famt/past_growth)/1000
    END as avg_size,
    NULL as gpunit,
  #Done matched
  policyno,

  amtreserve as statrsv,
  exhibit5,
  #Missmatch
  CASE 
    WHEN pbfrplan IS NULL THEN (
        CASE 
            WHEN LEFT(COALESCE(lengths, ''),1)='A' THEN CAST(RIGHT(lengths,2) AS INT64) - issAge
            WHEN term_plans IS NOT NULL THEN CAST(RIGHT(lengths,2) AS INT64)
            WHEN mort_char IS NULL THEN 0
            WHEN LPAD(CAST(valcode AS STRING), 5, '0') IN ("00595", "00695", "00725", "00794", "00795") THEN 65 - issAge
            WHEN LPAD(CAST(valcode AS STRING), 5, '0') ='00235' THEN 75 - issAge
            WHEN LPAD(CAST(valcode AS STRING), 5, '0') IN ("00519","00597", "P0302", "P0395", "0075N") THEN 85 - issAge
            WHEN pmdstssub ='ETI' THEN (
                DATE_DIFF(DATE_TRUNC(COALESCE(pmdstsexdt, CURRENT_DATE), MONTH),
                DATE_TRUNC(COALESCE(pmdissuedt, CURRENT_DATE), MONTH), YEAR)

                            )
            WHEN valcode NOT IN UNNEST(valCodePlans) AND term_plans IS NULL THEN (
                (CASE 
                    WHEN mort_char IN ("E", "F", "G", "H", "I", "J") THEN 99+1
                    WHEN mort_char ='A' THEN 103+1
                    WHEN mort_char ='B' THEN 98+1
                    WHEN mort_char ='C' THEN 100+1
                    WHEN mort_char ='D' THEN 120+1
                    WHEN mort_char ='K' THEN 120 +1
                    ELSE 0
                END) - issAge  
            )
            ELSE 0
        END
    ) 
    WHEN COALESCE(pbfrplan, '') IN ('40','19') THEN GREATEST(25 - issAge, 0)
    WHEN pbfrplan ='95' THEN GREATEST(65 - issAge, 0)
    ELSE 20
END AS insPD,

#Continue pfrplan is NULL side.

  pmdstssub as status,
  #matched
  NULL as premPayP,


  (final_famt-(final_famt/past_growth))/1000 as PUAUnitsIF,

  #matched
  
  CASE 
    WHEN past_growth = 0 OR (final_famt - (final_famt / past_growth)) = 0 THEN 0
    ELSE 
        ((final_famt - (final_famt / past_growth)) / 1000) 
        / 
        ((final_famt / past_growth) / 1000)
  END AS PUAUnit,


  IFNULL(bonus,0) as InitCPiBns,

  #diff - null 75507 records
  IFNULL(alfa_dbinc, 0) as DBPctInc,

  EXTRACT(YEAR FROM paidDate) as PaidToYr,
  EXTRACT(MONTH FROM paidDate) as PaidToMth,
  CASE WHEN pmdissuedt > DATE("2020-09-30") THEN "3"
        WHEN pmdissuedt> DATE("2018-09-30") THEN '2'
        ELSE "1"
        END as tranche,
 from field_prep;

    '''

In [6]:
annuity_query = f'''

select pmdplan, 
wrvage+CAST(wrvduratn as INT64)- (EXTRACT(YEAR FROM DATE("{valdate}"))-EXTRACT(YEAR FROM pmdissuedt))+
CASE WHEN EXTRACT(MONTH FROM pmdissuedt) > EXTRACT(MONTH FROM DATE("{valdate}")) THEN 1 ELSE 0 END as issAge,
wrvpbfsex,
NULL as Class,
"N" as tax_qualified,
"N" as simple_interest,
"1" as valcode,
CASE WHEN pmdissuedt > DATE("2019-10-01") THEN 2 ELSE 1 END as reins_flag,
0 as DBRider,
0 as FPRider,
0 as IntRider,
"M" as MygaDepType,
EXTRACT(YEAR FROM pmdissuedt) as IssYear,
EXTRACT(MONTH from pmdissuedt) as IssMonth,

"E" as newbus,
"1" as fixedLives,
policyno,
(SELECT premium2 from `lifetemp.annuity` a WHERE a.policyno = s.policyno and a.set_month ='{month}') as purchase_price,
0 as interest_earned,
0 as withdrawals,
(SELECT total2 from `lifetemp.annuity` a WHERE a.policyno = s.policyno and a.set_month ='{month}') as fund_value,
0 as gsmv,
CASE WHEN pmdplan IN ("PNA001", "PNA002", "PNA003", "PGA001", "PGA002", "POA005", "PGA005", "PGA006") THEN 0.03
      WHEN pmdplan IN ("PNA006", "PGA003", "POA006") THEN 0.025
      WHEN pmdplan IN ("PNA005", "PNA007", "PGA004", "PGA007") THEN 0.021
      WHEN pmdplan ='PNA004' THEN 0.0252
      END as year1_int_rate,
CASE WHEN pmdplan IN ("PNA001", "PNA002", "PNA003", "PGA001", "PGA002", "POA005", "PGA005", "PGA006") then 0.03
      WHEN pmdplan IN ("PNA006", "PGA003", "POA006") THEN 0.025
      WHEN pmdplan IN ("PNA005", "PNA007", "PGA004", "PGA007") THEN 0.021
      WHEN pmdplan ='PNA004' THEN 0.0252
      END as year2_int_rate,

NULL as year_1_5_int_rate,
NULL as year_6_10_int_rate,
NULL as year_11_15_int_rate,
NULL as year_16_20_int_rate,
0 as interest_basis,
1 as reins_pct,
0 as surrender_value_w_o_mva,
0 as surrender_value_w_mva,
final_rsv as stat_reserve,
0 as prior_tax_res,
0 as new_tax_res,
#NULL as prem_check_field,
#NULL as catmkt,
#NULL as mort_char,
#NULL as intchar,
120-(wrvage+CAST(wrvduratn as INT64)- (EXTRACT(YEAR FROM DATE("{valdate}"))-EXTRACT(YEAR FROM pmdissuedt))+
CASE WHEN EXTRACT(MONTH FROM pmdissuedt) > EXTRACT(MONTH FROM DATE("{valdate}")) THEN 1 ELSE 0 END) as term,
"a120" as maturity,
#NO issue state found so adding default value of 1
1 as other,
CASE WHEN pmdissuedt > DATE("2020-09-30") THEN "3"
        WHEN pmdissuedt> DATE("2018-09-30") THEN '2'
        ELSE "1"
        END as tranche,
wrvage+CAST(wrvduratn as INT64) as AA    
  from `life_prod.seriatim` s
WHERE set_month ='{month}' and exhibit5='5B';
'''

In [7]:
#Running life
result = client.query(query)
inforce_result = pd.DataFrame()

In [8]:
#Running annuity
result_ann = client.query(annuity_query)
annuity_inforce_result = pd.DataFrame()

In [9]:
result_ann

QueryJob<project=converge-database, location=US, id=fb1a3cf0-d3af-4aa4-a525-33797dee50b2>

In [10]:
ann_columns = [
    "pmdplan","issAge", "wrvpbfsex", "Class", "tax_qualified", "simple_interest", 
    "valcode", "reins_flag", "DBRider", "FPRider", "IntRider", "MygaDepType", 
    "IssYear", "IssMonth", "newbus", "fixedLives", "policyno", "purchase_price", 
    "interest_earned", "withdrawals", "fund_value", "gsmv", "year1_int_rate", 
    "year2_int_rate", "year_1_5_int_rate", "year_6_10_int_rate", "year_11_15_int_rate", 
    "year_16_20_int_rate", "interest_basis", "reins_pct", "surrender_value_w_o_mva", 
    "surrender_value_w_mva", "stat_reserve", "prior_tax_res", "new_tax_res", 
    "term", "maturity", "other", "tranche", "AA"
]
# Extract the rows from the query result and convert it to a list of tuples
data = [tuple(row) for row in result_ann]  # Assuming each row is iterable

# Create a DataFrame from the data and column headers
annuity_inforce_result = pd.DataFrame(data, columns=ann_columns)

In [11]:
annuity_inforce_result

,pmdplan,issAge,wrvpbfsex,Class,tax_qualified,simple_interest,valcode,reins_flag,DBRider,FPRider,...,surrender_value_w_o_mva,surrender_value_w_mva,stat_reserve,prior_tax_res,new_tax_res,term,maturity,other,tranche,AA
0,PGA002,66,M,None,N,N,1,2,0,0,...,0,0,1348.2,0,0,54,a120,1,3,70
1,PNA002,73,M,None,N,N,1,1,0,0,...,0,0,8462.2,0,0,47,a120,1,1,82
2,PNA002,93,F,None,N,N,1,2,0,0,...,0,0,1147.9,0,0,27,a120,1,3,97
3,PGA002,92,F,None,N,N,1,2,0,0,...,0,0,9876.1,0,0,28,a120,1,2,97
4,PNA002,97,F,None,N,N,1,1,0,0,...,0,0,3276.1,0,0,23,a120,1,2,104
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1713,PNA002,72,M,None,N,N,1,2,0,0,...,0,0,6741.4,0,0,48,a120,1,2,78
1714,PNA002,70,M,None,N,N,1,1,0,0,...,0,0,6681.8,0,0,50,a120,1,1,80
1715,PNA002,50,M,None,N,N,1,1,0,0,...,0,0,6462.3,0,0,70,a120,1,1,57
1716,PNA002,64,M,None,N,N,1,2,0,0,...,0,0,9076.0,0,0,56,a120,1,3,68


In [12]:
# Define the column headers based on the provided mapping
columns = [
    "plan", "term_plans","issAge","prem_check_field", "gender", "class", "catmkt", "mort_char", "valcode", "intchar", 
    "mode", "bnscreditMtd", "guar_non_guar", "uniq_policy_check", "term_wl", "issyr", 
    "issMonth", "newBusinessInd", "planver", "pmdplanver", "product2", "test", 
    "past_growth", "duratn", "alfa_dbinc", "pmdissuedt",
    "FixedLives", "PolCount", "face", "PremIf", "avg_size", "gpunit", "policyno", 
    "statrsv", "exhibi5","insPD","status","premPayP", "PUAUnitsIF", "PUAUnit", "InitCPiBns", "DBPctInc", 
    "PaidToYr", "PaidToMth", "tranche"
]


# Extract the rows from the query result and convert it to a list of tuples
data = [tuple(row) for row in result]  # Assuming each row is iterable

# Create a DataFrame from the data and column headers
inforce_result = pd.DataFrame(data, columns=columns)

In [13]:
inforce_result[
    (inforce_result['insPD'] == 0) & 
    (inforce_result['exhibi5'] == '5A')
]

,plan,term_plans,issAge,prem_check_field,gender,class,catmkt,mort_char,valcode,intchar,...,insPD,status,premPayP,PUAUnitsIF,PUAUnit,InitCPiBns,DBPctInc,PaidToYr,PaidToMth,tranche


In [14]:
inforce_result[inforce_result['policyno'] =='AA-0000061']

,plan,term_plans,issAge,prem_check_field,gender,class,catmkt,mort_char,valcode,intchar,...,insPD,status,premPayP,PUAUnitsIF,PUAUnit,InitCPiBns,DBPctInc,PaidToYr,PaidToMth,tranche
0,65,None,36.0,0,M,N,H,J,00999,D,...,64.0,RPU,None,0.0,0.0,0.0,0.0,2020,12,1


In [15]:
inforce_result['uniq_policy_check'].unique()

array([1, 0], dtype=int64)

In [16]:
# Fill missing `pp` values to avoid errors
inforce_result['prem_check_field'] = inforce_result['prem_check_field'].fillna('')  

In [17]:
# Extract numeric values from 'plan' (similar to RIGHT(plan,2))
def extract_numeric_plan(plan):
    match = re.search(r'(\d{2})$', str(plan))  # Get last two digits
    return int(match.group(1)) if match else None

# Extract numeric values from 'pp' (for conditions where pp ends in 'P')
def extract_numeric_pp(pp):
    match = re.search(r'(\d+)', str(pp))  # Extract digits
    return int(match.group(1)) if match else None

In [18]:
def calculate_prempay(row):
    if row['status'] in ["PRMPY", "WOD"]:
        # Extract numeric plan value
        plan_val = extract_numeric_plan(row['plan'])
        
        # Condition 1: If catmkt = 'P'
        if row['catmkt'] == 'P':
            result = plan_val
            #matched with excel
        # Condition 2: If catmkt <> 'P' and plan in (26, 20)
        elif row['catmkt'] != 'P' and plan_val in [26, 20]:
            result = 20
        #matched with excel
        # Condition 3: If `pp` starts with 'A'
        elif row['catmkt'] != 'P' and str(row['prem_check_field']).startswith('A'):
            pp_num = extract_numeric_plan(row['prem_check_field'])
            result = pp_num - row['issAge'] if pp_num else None
        
        # Condition 4: If `pp` ends with 'P'
        elif row['catmkt'] != 'P' and str(row['prem_check_field']).endswith('P'):
            result = extract_numeric_pp(row['prem_check_field'])
        
        else:
            result = row['insPD']  # Default case (like ELSE d.insPD in SQL)

        # Ensure insPD is the upper limit (like LEAST in SQL)
        insPD = pd.to_numeric(row['insPD'], errors='coerce')
        result_val = pd.to_numeric(result, errors='coerce')
        if pd.isna(result_val):  # If result is NaN, return insPD
            return insPD
        return np.nanmin([insPD, result_val])
    
    return 1  # Default value when status is not PRMPY or WOD

# Apply function to the DataFrame
inforce_result['prempay'] = inforce_result.apply(calculate_prempay, axis=1)


In [19]:
inforce_result.exhibi5

0         5A
1         5A
2         5A
3         5A
4         5A
          ..
101794    5A
101795    5A
101796    5A
101797    5A
101798    5A
Name: exhibi5, Length: 101799, dtype: object

In [20]:
#inforce_result_subset = inforce_result[inforce_result['exhibi5'].isin(['5A', '5B'])]
inforce_result_subset = inforce_result[inforce_result['exhibi5']=='5A']

In [21]:
inforce_result_subset.insPD.sum()

5639854.0

In [22]:
inforce_result_subset.prempay.sum()

499745.0

In [23]:
inforce_result_subset

,plan,term_plans,issAge,prem_check_field,gender,class,catmkt,mort_char,valcode,intchar,...,status,premPayP,PUAUnitsIF,PUAUnit,InitCPiBns,DBPctInc,PaidToYr,PaidToMth,tranche,prempay
0,65,None,36.0,0,M,N,H,J,00999,D,...,RPU,None,0.0,0.0,0.0,0.0,2020,12,1,1.0
1,8,None,64.0,0,M,N,H,J,00999,D,...,RPU,None,0.0,0.0,0.0,0.0,2001,6,1,1.0
2,10,None,46.0,0,M,N,H,J,00920,D,...,PRMPY,None,0.0,0.0,0.0,0.0,2026,4,1,54.0
3,8,None,57.0,0,F,N,H,J,00999,D,...,RPU,None,0.0,0.0,0.0,0.0,2004,11,1,1.0
4,8,None,72.0,0,F,N,H,J,00999,D,...,RPU,None,0.0,0.0,0.0,0.0,2004,9,1,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101794,10,None,26.0,0,F,N,P,J,00989,D,...,ETI,None,0.0,0.0,0.0,0.0,2029,7,1,1.0
101795,10,None,34.0,0,F,N,P,J,00989,D,...,ETI,None,0.0,0.0,0.0,0.0,2026,6,1,1.0
101796,10,None,46.0,0,F,N,H,J,00999,D,...,FPU,None,0.0,0.0,0.0,0.0,2055,12,1,1.0
101797,10,None,30.0,0,F,N,P,J,00989,D,...,ETI,None,0.0,0.0,0.0,0.0,2035,11,1,1.0


In [24]:
inforce_result.exhibi5[inforce_result['policyno'] =='AA-2100030']

7023    5A
Name: exhibi5, dtype: object

In [25]:
inforce_result_subset.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 92436 entries, 0 to 101798
Data columns (total 46 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   plan               92436 non-null  object 
 1   term_plans         338 non-null    object 
 2   issAge             92436 non-null  float64
 3   prem_check_field   92436 non-null  object 
 4   gender             92436 non-null  object 
 5   class              92436 non-null  object 
 6   catmkt             92436 non-null  object 
 7   mort_char          92436 non-null  object 
 8   valcode            92436 non-null  object 
 9   intchar            92436 non-null  object 
 10  mode               92436 non-null  object 
 11  bnscreditMtd       92436 non-null  object 
 12  guar_non_guar      92436 non-null  object 
 13  uniq_policy_check  92436 non-null  int64  
 14  term_wl            92436 non-null  object 
 15  issyr              92436 non-null  int64  
 16  issMonth           92

In [26]:
valdate = pd.to_datetime('2025-12-31')

In [27]:
inforce_result_subset['pmdissuedt'] = pd.to_datetime(inforce_result['pmdissuedt'], errors='coerce')

# Compute the difference in years
inforce_result_subset['AA'] = (valdate - inforce_result_subset['pmdissuedt']).dt.days / 365 + inforce_result_subset['issAge']

C:\Users\achoijin\AppData\Local\Temp\ipykernel_24392\1561437125.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inforce_result_subset['pmdissuedt'] = pd.to_datetime(inforce_result['pmdissuedt'], errors='coerce')
C:\Users\achoijin\AppData\Local\Temp\ipykernel_24392\1561437125.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inforce_result_subset['AA'] = (valdate - inforce_result_subset['pmdissuedt']).dt.days / 365 + inforce_result_subset['issAge']


In [28]:
issue_date = pd.to_datetime({
    'year': inforce_result_subset['issyr'] + inforce_result_subset['insPD'],
    'month': inforce_result_subset['issMonth'],
    'day': 28
})

inforce_result_subset['Age_filter'] = (issue_date-valdate ).dt.days / 365

C:\Users\achoijin\AppData\Local\Temp\ipykernel_24392\3451819740.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inforce_result_subset['Age_filter'] = (issue_date-valdate ).dt.days / 365


In [29]:
inforce_result_subset[inforce_result_subset['policyno']=='AA-0000061']

,plan,term_plans,issAge,prem_check_field,gender,class,catmkt,mort_char,valcode,intchar,...,PUAUnitsIF,PUAUnit,InitCPiBns,DBPctInc,PaidToYr,PaidToMth,tranche,prempay,AA,Age_filter
0,65,None,36.0,0,M,N,H,J,00999,D,...,0.0,0.0,0.0,0.0,2020,12,1,1.0,62.016438,38.10137


In [30]:
inforce_result_subset['gpunit'] = np.where(
    inforce_result_subset['policyno'] == 0,
    0,
    inforce_result_subset['PremIf'] / inforce_result_subset['avg_size']
)

C:\Users\achoijin\AppData\Local\Temp\ipykernel_24392\622961976.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inforce_result_subset['gpunit'] = np.where(


In [33]:
output_path = 'inforce_results/'

In [34]:
inforce_result_subset.to_excel(f'{output_path}req.xlsx', index=False)

In [35]:
aa100 = inforce_result_subset[inforce_result_subset['Age_filter']<0].reset_index()

In [36]:
aa100

,index,plan,term_plans,issAge,prem_check_field,gender,class,catmkt,mort_char,valcode,...,PUAUnitsIF,PUAUnit,InitCPiBns,DBPctInc,PaidToYr,PaidToMth,tranche,prempay,AA,Age_filter
0,99,40,40,2.0,A25,F,N,H,J,00945,...,0.0,0.0,0.0,0.000,2026,4,1,23.0,27.454795,-2.430137
1,100,40,40,0.0,A25,M,N,H,J,00945,...,0.0,0.0,0.0,0.000,2026,4,1,25.0,25.454795,-0.427397
2,151,19,19,0.0,A25,F,N,H,H,00719,...,0.0,0.0,0.0,0.000,2098,1,1,1.0,25.972603,-0.923288
3,981,40,40,4.0,A25,M,N,H,J,00945,...,0.0,0.0,0.0,0.000,2026,4,1,21.0,27.852055,-2.764384
4,1237,40,40,3.0,A25,F,N,H,J,00945,...,0.0,0.0,0.0,0.000,2026,4,1,22.0,26.098630,-1.008219
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
470,89131,GTIA01,None,96.0,,F,N,P,J,090D1,...,NaN,NaN,0.0,0.018,2045,5,3,1.0,100.654795,-0.594521
471,92185,GTIA01,None,97.0,,F,N,P,J,090D1,...,NaN,NaN,0.0,0.018,2044,10,3,1.0,101.213699,-1.175342
472,99619,GTIA01,None,97.0,,F,N,P,J,090D1,...,NaN,NaN,0.0,0.018,2045,7,3,1.0,100.454795,-0.427397
473,99839,GTIA01,None,98.0,,F,N,P,J,090D1,...,NaN,NaN,0.0,0.018,2044,7,3,1.0,101.430137,-1.427397


In [ ]:
# inforce_result_subset.to_gbq("converge-database.life_prod.life_inforce_build_db",
#                  if_exists='append',
#                   table_schema=None,
#                  project_id="converge-database")

In [37]:
#subset for aa<100, HS/FE
subset1 = inforce_result_subset[
    (inforce_result_subset['Age_filter'] > 0) &
    (inforce_result_subset['catmkt'].isin(['H', 'F']))
].reset_index()

In [38]:
subset2 =inforce_result_subset[
    (inforce_result_subset['Age_filter'] > 0) &
    (inforce_result_subset['catmkt']=='P')
].reset_index()

In [39]:
subset1

,index,plan,term_plans,issAge,prem_check_field,gender,class,catmkt,mort_char,valcode,...,PUAUnitsIF,PUAUnit,InitCPiBns,DBPctInc,PaidToYr,PaidToMth,tranche,prempay,AA,Age_filter
0,0,65,None,36.0,0,M,N,H,J,00999,...,0.0,0.0,0.0,0.0,2020,12,1,1.0,62.016438,38.101370
1,1,8,None,64.0,0,M,N,H,J,00999,...,0.0,0.0,0.0,0.0,2001,6,1,1.0,90.008219,10.082192
2,2,10,None,46.0,0,M,N,H,J,00920,...,0.0,0.0,0.0,0.0,2026,4,1,54.0,72.016438,28.095890
3,3,8,None,57.0,0,F,N,H,J,00999,...,0.0,0.0,0.0,0.0,2004,11,1,1.0,83.016438,17.087671
4,4,8,None,72.0,0,F,N,H,J,00999,...,0.0,0.0,0.0,0.0,2004,9,1,1.0,98.016438,2.076712
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51484,101789,10,None,30.0,0,F,N,H,H,00789,...,0.0,0.0,0.0,0.0,2027,1,1,1.0,74.361644,1.076712
51485,101790,10,None,21.0,0,F,N,H,H,00789,...,0.0,0.0,0.0,0.0,2030,1,1,1.0,59.778082,4.079452
51486,101791,10,None,37.0,0,F,N,H,H,00799,...,0.0,0.0,0.0,0.0,2050,8,1,1.0,75.443836,24.673973
51487,101792,10,None,21.0,0,F,N,H,H,00789,...,0.0,0.0,0.0,0.0,2031,1,1,1.0,58.189041,5.079452


In [40]:
subset2

,index,plan,term_plans,issAge,prem_check_field,gender,class,catmkt,mort_char,valcode,...,PUAUnitsIF,PUAUnit,InitCPiBns,DBPctInc,PaidToYr,PaidToMth,tranche,prempay,AA,Age_filter
0,10,10,None,18.0,0,M,N,P,J,00989,...,0.0,0.0,0.0,0.0,2047,5,1,1.0,43.931507,21.419178
1,11,10,None,16.0,0,M,N,P,J,00989,...,0.0,0.0,0.0,0.0,2048,3,1,1.0,41.931507,22.254795
2,12,10,None,32.0,0,M,N,P,J,00989,...,0.0,0.0,0.0,0.0,2033,10,1,1.0,57.931507,7.830137
3,13,10,None,24.0,0,M,N,P,J,00989,...,0.0,0.0,0.0,0.0,2044,3,1,1.0,49.931507,18.252055
4,33,10,None,9.0,0,F,N,P,J,00989,...,0.0,0.0,0.0,0.0,2034,2,1,1.0,34.931507,8.167123
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40467,101793,10,None,37.0,0,F,N,P,J,00989,...,0.0,0.0,0.0,0.0,2030,7,1,1.0,73.441096,4.575342
40468,101794,10,None,26.0,0,F,N,P,J,00989,...,0.0,0.0,0.0,0.0,2029,7,1,1.0,60.608219,3.575342
40469,101795,10,None,34.0,0,F,N,P,J,00989,...,0.0,0.0,0.0,0.0,2026,6,1,1.0,67.353425,0.490411
40470,101797,10,None,30.0,0,F,N,P,J,00989,...,0.0,0.0,0.0,0.0,2035,11,1,1.0,58.350685,9.915068


In [41]:
final_column = ['plan', 'issAge', 'gender', 'class', 'catmkt', 'mort_char', 'intchar', 'mode', 'bnscreditMtd', 
           'guar_non_guar', 'uniq_policy_check', 'term_wl', 'issyr', 'issMonth', 'newBusinessInd', 'FixedLives', 'PolCount', 'face', 'PremIf', 'avg_size', 'gpunit','policyno'
               ,'statrsv', 'insPD', 'status', 'prempay', 'PUAUnitsIF','PUAUnit', 'InitCPiBns', 'DBPctInc', 'PaidToYr', 'PaidToMth', 'tranche']

In [42]:
# FINAL Reorder and subset columns 
subset1 = subset1[final_column]
subset2 = subset2[final_column]
aa100 = aa100[final_column]

In [43]:
aa100.to_excel(f'{output_path}aa100_inforce_output_{month}.xlsx', index=False)
subset1.to_excel(f'{output_path}h_and_f_inforce_output_{month}.xlsx', index=False)
subset2.to_excel(f'{output_path}p_inforce_output_{month}.xlsx', index=False)


In [44]:
annuity_inforce_result.to_excel(f'{output_path}ann_inforce_{month}.xlsx', index=False)